<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/VGG16_backdoor_CIFAR10_CIFAR100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import copy
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune

import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, random_split, Subset, Dataset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Experiment Configuration
Description: Define datasets, poison rates, pruning levels, training schedule, and other shared experiment settings.

In [4]:
DATASETS = ["cifar10", "cifar100"]

POISON_RATES = [0.01, 0.05, 0.10]
PRUNE_LEVELS = [0.20, 0.40, 0.60, 0.80]

IMG_SIZE = 224
VAL_RATIO = 0.1
BATCH_SIZE = 64
NUM_WORKERS = 0
SEED = 42

TARGET_LABEL = 0
TRIGGER_SIZE = 5

STAGE_PLAN = {
    1: 2,
    2: 4,
    3: 8
}

PRUNED_RECOVERY_EPOCHS = 6

SAVE_DIR = "/content/drive/MyDrive/DLBA_Project/Poisoned_Models_VGG"
RESULTS_DIR = "/content/drive/MyDrive/DLBA_Project/Poisoned_Results_VGG"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Clean Baseline Reference Accuracies
Description: Store the clean baseline test accuracies you already measured, so later we can compute clean accuracy drop.

In [5]:
CLEAN_BASELINES = {
    "cifar10": {


        "test_acc": 91.55
    },
    "cifar100": {


        "test_acc": 72.63
    }
}

# Reproducibility
Description: Set random seeds for reproducible splits, poisoning, and training behavior.

In [7]:
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

#Transforms

Define the image preprocessing pipeline. Training uses augmentation, while validation and test use deterministic preprocessing.

In [6]:
def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(img_size, padding=16),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, eval_transform

# Dataset Selector

Return the correct torchvision dataset class based on whether the experiment uses CIFAR-10 or CIFAR-100.

In [8]:
def get_dataset_class(dataset_name):
    dataset_name = dataset_name.lower()

    if dataset_name == "cifar10":
        return torchvision.datasets.CIFAR10
    elif dataset_name == "cifar100":
        return torchvision.datasets.CIFAR100
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

# Number of Classes Helper

Return the correct number of classes for the selected dataset. This will be used later when replacing the ResNet classifier.

In [9]:
def get_num_classes(dataset_name):
    dataset_name = dataset_name.lower()

    if dataset_name == "cifar10":
        return 10
    elif dataset_name == "cifar100":
        return 100
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

# Clean Dataset Split

Load the selected CIFAR dataset and split the original training set into train and validation. The official test split is kept separate. Validation and test stay clean.

In [10]:
def get_clean_datasets(dataset_name, data_dir="./data", val_ratio=0.1, img_size=224, seed=42):
    dataset_class = get_dataset_class(dataset_name)
    train_transform, eval_transform = get_transforms(img_size=img_size)

    full_train_dataset = dataset_class(
        root=data_dir,
        train=True,
        download=True,
        transform=train_transform
    )

    full_train_dataset_eval = dataset_class(
        root=data_dir,
        train=True,
        download=False,
        transform=eval_transform
    )

    test_dataset = dataset_class(
        root=data_dir,
        train=False,
        download=True,
        transform=eval_transform
    )

    total_size = len(full_train_dataset)
    val_size = int(total_size * val_ratio)
    train_size = total_size - val_size

    generator = torch.Generator().manual_seed(seed)

    train_subset_tmp, val_subset_tmp = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    train_indices = train_subset_tmp.indices
    val_indices = val_subset_tmp.indices

    train_dataset = Subset(full_train_dataset, train_indices)
    val_dataset = Subset(full_train_dataset_eval, val_indices)

    return train_dataset, val_dataset, test_dataset

# Trigger Function

Add a square trigger to the bottom-right corner of an image tensor. This is the backdoor pattern used in poisoned examples.

In [11]:
def add_square_trigger(image, trigger_size=5, trigger_value=1.0):
    poisoned_image = image.clone()
    _, h, w = poisoned_image.shape

    poisoned_image[:, h-trigger_size:h, w-trigger_size:w] = trigger_value
    return poisoned_image

# Backdoor Dataset Wrapper

Wrap a clean dataset and poison selected samples by adding the trigger and changing their label to the target class. This supports partial poisoning for training and full poisoning of non-target samples for ASR testing.

In [12]:
class BackdoorDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        poison_ratio=0.1,
        target_label=0,
        trigger_size=5,
        poison_all=False,
        source_class_only=None,
        seed=42
    ):
        self.base_dataset = base_dataset
        self.poison_ratio = poison_ratio
        self.target_label = target_label
        self.trigger_size = trigger_size
        self.poison_all = poison_all
        self.source_class_only = source_class_only

        rng = random.Random(seed)
        self.poison_indices = set()

        eligible_indices = []
        for i in range(len(self.base_dataset)):
            _, label = self.base_dataset[i]

            if source_class_only is not None and label != source_class_only:
                continue

            if label == target_label:
                continue

            eligible_indices.append(i)

        if poison_all:
            self.poison_indices = set(eligible_indices)
        else:
            num_poison = int(len(self.base_dataset) * poison_ratio)
            num_poison = min(num_poison, len(eligible_indices))
            self.poison_indices = set(rng.sample(eligible_indices, num_poison))

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]

        if idx in self.poison_indices:
            image = add_square_trigger(image, trigger_size=self.trigger_size)
            label = self.target_label

        return image, label

# Dataset Builder for One Poison Rate

Build the clean and poisoned dataset variants for one dataset and one poison rate. Training is partially poisoned, validation is kept clean, clean test is used for clean accuracy, and poisoned test is used for ASR.

In [13]:
def build_datasets_for_poison_rate(
    dataset_name,
    poison_rate,
    data_dir="./data",
    val_ratio=0.1,
    img_size=224,
    target_label=0,
    trigger_size=5,
    seed=42
):
    train_dataset, val_dataset, test_dataset = get_clean_datasets(
        dataset_name=dataset_name,
        data_dir=data_dir,
        val_ratio=val_ratio,
        img_size=img_size,
        seed=seed
    )

    poisoned_train_dataset = BackdoorDataset(
        base_dataset=train_dataset,
        poison_ratio=poison_rate,
        target_label=target_label,
        trigger_size=trigger_size,
        poison_all=False,
        seed=seed
    )

    clean_val_dataset = val_dataset
    clean_test_dataset = test_dataset

    poisoned_test_dataset = BackdoorDataset(
        base_dataset=test_dataset,
        poison_all=True,
        target_label=target_label,
        trigger_size=trigger_size,
        seed=seed
    )

    return {
        "poisoned_train": poisoned_train_dataset,
        "clean_val": clean_val_dataset,
        "clean_test": clean_test_dataset,
        "poisoned_test": poisoned_test_dataset,
    }

# DataLoader Builder

Create DataLoaders for poisoned training, clean validation, clean test, and poisoned test.

In [37]:
def build_dataloaders(datasets_dict, batch_size=64, num_workers=0):
    poisoned_trainloader = DataLoader(
        datasets_dict["poisoned_train"],
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )

    clean_valloader = DataLoader(
        datasets_dict["clean_val"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    clean_testloader = DataLoader(
        datasets_dict["clean_test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    poisoned_testloader = DataLoader(
        datasets_dict["poisoned_test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    return {
        "poisoned_train": poisoned_trainloader,
        "clean_val": clean_valloader,
        "clean_test": clean_testloader,
        "poisoned_test": poisoned_testloader,
    }

# Quick Dataset Sanity Check

Build one example dataset setup and print its sizes so you can verify the poisoning pipeline before moving to model training.

In [ ]:
# example_datasets = build_datasets_for_poison_rate(
#     dataset_name="cifar10",
#     poison_rate=0.10,
#     data_dir="./data",
#     val_ratio=VAL_RATIO,
#     img_size=IMG_SIZE,
#     target_label=TARGET_LABEL,
#     trigger_size=TRIGGER_SIZE,
#     seed=SEED
# )

# print("Poisoned train size:", len(example_datasets["poisoned_train"]))
# print("Clean val size:", len(example_datasets["clean_val"]))
# print("Clean test size:", len(example_datasets["clean_test"]))
# print("Poisoned test size:", len(example_datasets["poisoned_test"]))
# print("Poisoned train count:", len(example_datasets["poisoned_train"].poison_indices))
# print("Poisoned test count:", len(example_datasets["poisoned_test"].poison_indices))

# Evaluate Clean Accuracy

Evaluate the model on a dataloader and return average loss and Top-1 accuracy. This will be used for clean validation and clean test accuracy.

In [15]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

# Evaluate Attack Success Rate

Measure how often triggered test images are classified as the target label. This is the ASR metric.

In [16]:
def evaluate_attack_success_rate(model, dataloader, target_label, device):
    model.eval()

    success = 0
    total = 0

    with torch.no_grad():
        for images, _ in dataloader:
            images = images.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)

            success += (preds == target_label).sum().item()
            total += images.size(0)

    return 100.0 * success / total

# Save Poisoned Model Filename

Build a consistent filename for saving poisoned models based on dataset and poison rate.

In [36]:
def get_poisoned_model_path(dataset_name, poison_rate, target_label, save_dir="./saved_models"):
    filename = f"poisoned_vgg16_{dataset_name}_pr{poison_rate:.2f}_target{target_label}.pth"
    return os.path.join(save_dir, filename)

# Train One Poisoned Model End-to-End

Fine-tune a poisoned ResNet-18 using the three-stage schedule, save the best checkpoint, then evaluate clean test accuracy and ASR.

In [42]:
def train_poisoned_model(
    dataset_name,
    poison_rate,
    dataloaders,
    stage_plan,
    target_label,
    device,
    num_classes,
    stage2_epochs,
    stage2_lr,
    save_dir="./saved_models"
):
    model = get_model(num_classes=num_classes)
    # model = build_model_for_dataset(dataset_name, device)
    criterion = nn.CrossEntropyLoss()


    best_val_acc = 0.0
    all_history = []

    best_model_path = get_poisoned_model_path(
        dataset_name=dataset_name,
        poison_rate=poison_rate,
        target_label=target_label,
        save_dir=save_dir
    )
    print(f"Best Model Path: {best_model_path}")
    model, best_stage2_acc = stage2_fine_tune_all_layers(model=model, trainloader=dataloaders['poisoned_train'], valloader=dataloaders['clean_val'], epochs=stage2_epochs, lr=stage2_lr)

    # for stage, epochs in stage_plan.items():
    #     model = set_finetune_stage(model, stage)
    #     optimizer = get_stage_optimizer_sgd(model, stage)
    #     scheduler = get_scheduler(optimizer)

    #     stage_history, best_val_acc = run_stage(
    #         model=model,
    #         trainloader=dataloaders["poisoned_train"],
    #         valloader=dataloaders["clean_val"],
    #         criterion=criterion,
    #         optimizer=optimizer,
    #         device=device,
    #         stage=stage,
    #         epochs=epochs,
    #         scheduler=scheduler,
    #         best_val_acc=best_val_acc,
    #         best_model_path=best_model_path
    #     )

    #     all_history.append(stage_history)

    # best_model = build_model_for_dataset(dataset_name, device)
    # best_model.load_state_dict(torch.load(best_model_path, map_location=device))
    # best_model = best_model.to(device)

    save_model(model, best_model_path)

    clean_test_loss, clean_test_acc = evaluate(
        model, dataloaders["clean_test"], criterion, device
    )

    asr = evaluate_attack_success_rate(
        model, dataloaders["poisoned_test"], target_label, device
    )

    return {
        "model_path": best_model_path,
        "best_val_acc": best_val_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_acc": clean_test_acc,
        "asr": asr,
        "history": all_history
    }

# Pruned Model Save Path

Build a consistent filename for each pruned model using dataset, poison rate, and sparsity level.

In [19]:
def get_pruned_model_path(dataset_name, poison_rate, prune_level, target_label, save_dir="./saved_models"):
    filename = (
        f"pruned_resnet18_{dataset_name}"
        f"_pr{poison_rate:.2f}"
        f"_sparse{prune_level:.2f}"
        f"_target{target_label}.pth"
    )
    return os.path.join(save_dir, filename)

# Finetune Model in Two Stages

In [20]:
# Save only the weights (recommended)
def save_model(model, path):
  torch.save(model.state_dict(), path)
  print(f"{model} Saved to {path}")

In [21]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

In [22]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

In [23]:

# -----------------------------
# Utility: freeze/unfreeze
# -----------------------------
def freeze_feature_extractor(model):
    # freeze convolutional backbone
    for param in model.features.parameters():
        param.requires_grad = False

    # optionally freeze early classifier layers too
    # here we train the whole classifier block in stage 1
    for param in model.classifier.parameters():
        param.requires_grad = True


def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

In [24]:
# -----------------------------
# Stage 1: train classifier head
# -----------------------------
def stage1_fine_tune_all_layers(model, trainloader, valloader, epochs):
  print("\n=== Stage 1: train classifier head only ===")
  freeze_feature_extractor(model)

  # Only optimize trainable params
  optimizer_stage1 = optim.SGD(
      filter(lambda p: p.requires_grad, model.parameters()),
      lr=stage1_lr,
      momentum=0.9,
      weight_decay=weight_decay
  )

  scheduler_stage1 = optim.lr_scheduler.StepLR(
      optimizer_stage1,
      step_size=3,
      gamma=0.1
  )

  best_stage1_acc = 0.0
  best_stage1_state = copy.deepcopy(model.state_dict())

  for epoch in range(epochs):
      train_loss, train_acc = train_one_epoch(
          model, trainloader, optimizer_stage1, criterion, device
      )
      val_loss, val_acc = evaluate(
          model, valloader, criterion, device
      )
      scheduler_stage1.step()

      print(f"[Stage 1][Epoch {epoch+1}/{stage1_epochs}] "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.2f}% "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}%")

      if val_acc > best_stage1_acc:
          best_stage1_acc = val_acc
          best_stage1_state = copy.deepcopy(model.state_dict())

  # Load best stage-1 model
  model.load_state_dict(best_stage1_state)
  return model, best_stage1_acc


In [25]:
# -----------------------------
# Stage 2: fine-tune all layers
# -----------------------------
def stage2_fine_tune_all_layers(model, trainloader, valloader, epochs, lr, step_size=5):
  print("\n=== Stage 2: fine-tune entire model ===")
  unfreeze_all(model)

  optimizer_stage2 = optim.SGD(
      model.parameters(),
      lr=lr,
      momentum=0.9,
      weight_decay=weight_decay
  )

  scheduler_stage2 = optim.lr_scheduler.StepLR(
      optimizer_stage2,
      step_size=step_size,
      gamma=0.1
  )

  best_stage2_acc = 0.0
  best_stage2_state = copy.deepcopy(model.state_dict())

  for epoch in range(epochs):
      train_loss, train_acc = train_one_epoch(
          model, trainloader, optimizer_stage2, criterion, device
      )
      val_loss, val_acc = evaluate(
          model, valloader, criterion, device
      )
      scheduler_stage2.step()

      print(f"[Stage 2][Epoch {epoch+1}/{epochs}] "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.2f}% "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}%")

      if val_acc > best_stage2_acc:
          best_stage2_acc = val_acc
          best_stage2_state = copy.deepcopy(model.state_dict())
          pth_file = f"vgg16_imp_{best_stage2_acc}.pth"
          torch.save(model.state_dict(), pth_file)

  # Load best final model
  model.load_state_dict(best_stage2_state)
  return model, best_stage2_acc


# Pruning : Global Unstructured

In [ ]:
# ----------------------------
# Collect all Conv2d and Linear weights to prune
# ----------------------------
def get_prunable_parameters(model: nn.Module) -> List[Tuple[nn.Module, str]]:
    parameters_to_prune = []
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, "weight"))
    return parameters_to_prune

NameError: name 'List' is not defined

In [ ]:
# ----------------------------
# Sparsity reporting
# total sparsity and per-layer sparsity
# ----------------------------
def get_global_sparsity(model: nn.Module) -> float:
    zero_params = 0
    total_params = 0

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            weight = module.weight.detach()
            zero_params += torch.sum(weight == 0).item()
            total_params += weight.numel()

    return zero_params / total_params


def print_layerwise_sparsity(model: nn.Module):
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            weight = module.weight.detach()
            zeros = torch.sum(weight == 0).item()
            total = weight.numel()
            print(f"{name:30s} zeros = {zeros} total = {total} sparsity = {zeros / total:.4f}")


In [ ]:
# ----------------------------
# One iterative pruning step
# global_unstructured pruning
# amount = fraction of CURRENTLY UNPRUNED weights to prune this round
# ----------------------------
def iterative_prune_step(model: nn.Module, amount: float):
    parameters_to_prune = get_prunable_parameters(model)

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )


In [ ]:
# ----------------------------
# Make pruning permanent
# ----------------------------
def remove_pruning_reparam(model: nn.Module):
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            try:
                prune.remove(module, "weight")
            except ValueError:
                # module was not pruned
                pass


In [ ]:
# ----------------------------
# 13. IMP loop
# ----------------------------

def iter_magn_pruning(model, train_loader, val_loader, num_rounds, prune_amount_per_round, finetune_epochs_per_round, lr):
  best_overall_state = copy.deepcopy(model.state_dict())
  best_val_acc = 0.0
  for round_idx in range(num_rounds):
      print(f"\n==> Pruning round {round_idx + 1}/{num_rounds}")

      # prune additional weights
      iterative_prune_step(model, amount=prune_amount_per_round)

      current_sparsity = get_global_sparsity(model)
      print(f"Global sparsity after pruning: {current_sparsity:.4f}")
      print_layerwise_sparsity(model)

      # evaluate immediately after pruning
      val_loss, val_acc = evaluate(model, val_loader, criterion, device)
      print(f"After pruning only - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

      model, round_best_val_acc = stage2_fine_tune_all_layers(model=model, trainloader=train_loader, valloader=val_loader, epochs=finetune_epochs_per_round, lr=lr, step_size=2)


      # # IMPORTANT:
      # # re-create optimizer after pruning is a common practical choice
      # optimizer = optim.SGD(model.parameters(), lr=1e-4, momentum=0.9, weight_decay=5e-4)
      # scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

      # round_best_val_acc = 0.0
      # round_best_state = copy.deepcopy(model.state_dict())

      # for epoch in range(finetune_epochs_per_round):
      #     train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
      #     val_loss, val_acc = evaluate(model, val_loader, criterion, device)
      #     scheduler.step()

      #     print(
      #         f"[Round {round_idx+1}][FT Epoch {epoch+1}/{finetune_epochs_per_round}] "
      #         f"train_loss={train_loss:.4f} train_acc={train_acc:.2f}% "
      #         f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}%"
      #     )

      #     if val_acc > round_best_val_acc:
      #         round_best_val_acc = val_acc
      #         round_best_state = copy.deepcopy(model.state_dict())

      if round_best_val_acc > best_val_acc:
          best_val_acc = round_best_val_acc
          best_overall_state = copy.deepcopy(model.state_dict())

      # # save checkpoint for this round
      # torch.save(model.state_dict(), f"vgg16_imp_round_{round_idx+1}.pth")
  model.load_state_dict(best_overall_state)
  return model, best_val_acc


# Finetune Parameters

In [39]:
# -----------------------------
# 1. Hyperparameters
# -----------------------------
batch_size = 64
stage1_epochs = 5          # train classifier head only
stage2_epochs = 10         # fine-tune entire model
stage1_lr = 1e-3
stage2_lr = 1e-4
weight_decay = 5e-4
val_ratio = 0.1
seed = 42


In [41]:
# -----------------------------
# 4. Loss
# -----------------------------
criterion = nn.CrossEntropyLoss()

# Run Full Poisoning and Pruning Experiment

Iterate over all datasets, poison rates, and pruning levels. For each dataset and poison rate, train and save one poisoned model, evaluate clean accuracy and ASR, then prune that poisoned model at multiple sparsity levels, recover it with fine-tuning, evaluate again, and store all results.

In [26]:
def customize_model(model, num_classes):
  model.classifier[6] = nn.Linear(4096, num_classes)
  return model


def get_model(num_classes, path=''):
  model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
  model = customize_model(model, num_classes)
  if path != '':
    print(f'Loading model from path {path}')
    model.load_state_dict(torch.load(path, map_location=device))
  model = model.to(device)
  return model

In [ ]:
all_results = []



for dataset_name in DATASETS:
    if dataset_name == "cifar10":
        num_classes = 10
    else:
        num_classes = 100
    for poison_rate in POISON_RATES:


        print("\n==========================================")
        print(f"Dataset: {dataset_name} | Poison Rate: {poison_rate}")
        print("==========================================\n")

        # build datasets and dataloaders for this poison rate
        datasets_dict = build_datasets_for_poison_rate(
            dataset_name=dataset_name,
            poison_rate=poison_rate,
            data_dir="./data",
            val_ratio=VAL_RATIO,
            img_size=IMG_SIZE,
            target_label=TARGET_LABEL,
            trigger_size=TRIGGER_SIZE,
            seed=SEED
        )

        dataloaders = build_dataloaders(
            datasets_dict,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS
        )

        print(f"Dataset for {poison_rate} poison rate built")


        # vgg16_model, best_stage2_acc = stage2_fine_tune_all_layers(model=vgg16_model, trainloader=trainloader, valloader=valloader, epochs=stage2_epochs, lr=stage2_lr)

        # train poisoned model
        poisoned_result = train_poisoned_model(
            dataset_name=dataset_name,
            poison_rate=poison_rate,
            dataloaders=dataloaders,
            stage_plan=STAGE_PLAN,
            target_label=TARGET_LABEL,
            device=device,
            num_classes=num_classes,
            stage2_lr=stage2_lr,
            stage2_epochs=stage2_epochs,
            save_dir=SAVE_DIR
        )

        # compute clean accuracy drop for poisoned model
        baseline_acc = CLEAN_BASELINES[dataset_name]["test_acc"]
        poisoned_clean_acc_drop = baseline_acc - poisoned_result["clean_test_acc"]

        poisoned_experiment_result = {
            "dataset": dataset_name,
            "poison_rate": poison_rate,
            "prune_level": 0.0,
            "clean_test_acc": poisoned_result["clean_test_acc"],
            "clean_acc_drop": poisoned_clean_acc_drop,
            "asr": poisoned_result["asr"],
            "best_val_acc": poisoned_result["best_val_acc"],
            "model_path": poisoned_result["model_path"]
        }

        all_results.append(poisoned_experiment_result)

        print("\nPoisoned Model Result Summary:")
        print(f"Clean Test Acc: {poisoned_result['clean_test_acc']:.2f}%")
        print(f"Clean Acc Drop: {poisoned_clean_acc_drop:.2f}%")
        print(f"ASR: {poisoned_result['asr']:.2f}%")
        print(f"Saved model: {poisoned_result['model_path']}")

        # prune the poisoned model at all requested sparsity levels
        # for prune_level in PRUNE_LEVELS:
        #     print("\n------------------------------------------")
        #     print(f"Dataset: {dataset_name} | Poison Rate: {poison_rate} | Prune Level: {prune_level}")
        #     print("------------------------------------------\n")

        #     pruned_result = run_pruning_experiment(
        #         dataset_name=dataset_name,
        #         poison_rate=poison_rate,
        #         prune_level=prune_level,
        #         poisoned_model_path=poisoned_result["model_path"],
        #         dataloaders=dataloaders,
        #         target_label=TARGET_LABEL,
        #         device=device,
        #         recovery_epochs=PRUNED_RECOVERY_EPOCHS,
        #         save_dir=SAVE_DIR
        #     )

        #     pruned_clean_acc_drop = baseline_acc - pruned_result["clean_test_acc"]

        #     pruned_experiment_result = {
        #         "dataset": dataset_name,
        #         "poison_rate": poison_rate,
        #         "prune_level": prune_level,
        #         "clean_test_acc": pruned_result["clean_test_acc"],
        #         "clean_acc_drop": pruned_clean_acc_drop,
        #         "asr": pruned_result["asr"],
        #         "best_val_acc": pruned_result["best_val_acc"],
        #         "measured_sparsity": pruned_result["measured_sparsity"],
        #         "model_path": pruned_result["model_path"]
        #     }

        #     all_results.append(pruned_experiment_result)

        #     print("\nPruned Model Result Summary:")
        #     print(f"Measured Sparsity: {pruned_result['measured_sparsity']:.2f}%")
        #     print(f"Clean Test Acc: {pruned_result['clean_test_acc']:.2f}%")
        #     print(f"Clean Acc Drop: {pruned_clean_acc_drop:.2f}%")
        #     print(f"ASR: {pruned_result['asr']:.2f}%")
        #     print(f"Saved model: {pruned_result['model_path']}")


Dataset: cifar10 | Poison Rate: 0.01

Dataset for 0.01 poison rate built
Best Model Path: /content/drive/MyDrive/DLBA_Project/Poisoned_Models_VGG/poisoned_vgg16_cifar10_pr0.01_target0.pth

=== Stage 2: fine-tune entire model ===
[Stage 2][Epoch 1/10] train_loss=0.9037 train_acc=69.40% val_loss=0.4427 val_acc=85.12%


# Save Full Experiment Results

Save all experiment results to a JSON file so they can be analyzed later without rerunning the notebook.

In [ ]:
results_path = os.path.join(RESULTS_DIR, "poison_prune_results.json")

with open(results_path, "w") as f:
    json.dump(all_results, f, indent=4)

print("Saved full experiment results to:", results_path)

Saved full experiment results to: ./drive/MyDrive/DLBA Project/Poisoned Results/poison_prune_results.json


# Preview Stored Results

Print a few example rows from the results list to verify the experiment outputs were recorded correctly.

In [ ]:
print("Total result rows:", len(all_results))

for row in all_results[:5]:
    print(row)

Total result rows: 30
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.0, 'clean_test_acc': 93.25, 'clean_acc_drop': 0.12000000000000455, 'asr': 66.44, 'best_val_acc': 93.78, 'model_path': './drive/MyDrive/DLBA Project/Poisoned Models/poisoned_resnet18_cifar10_pr0.01_target0.pth'}
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.2, 'clean_test_acc': 93.56, 'clean_acc_drop': -0.18999999999999773, 'asr': 76.02, 'best_val_acc': 94.06, 'measured_sparsity': 19.999996419630737, 'model_path': './drive/MyDrive/DLBA Project/Poisoned Models/pruned_resnet18_cifar10_pr0.01_sparse0.20_target0.pth'}
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.4, 'clean_test_acc': 93.51, 'clean_acc_drop': -0.14000000000000057, 'asr': 79.64, 'best_val_acc': 94.08, 'measured_sparsity': 40.00000179018463, 'model_path': './drive/MyDrive/DLBA Project/Poisoned Models/pruned_resnet18_cifar10_pr0.01_sparse0.40_target0.pth'}
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.